In [ ]:
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

all_links = set()  # Dùng set để tránh trùng lặp

for page in range(1, 41):  # Duyệt từ trang 1 đến 40
    url = f"https://www.alphabooks.vn/collections/all?q=&page={page}&view=grid"
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Lỗi khi truy cập {url}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    # Tìm các thẻ div có class 'item_product_main'
    items = soup.find_all("div", class_="item_product_main")

    # Lấy link từ thẻ a trong mỗi item
    links = {f"https://www.alphabooks.vn{item.find('a')['href']}" for item in items if item.find("a")}

    # Thêm vào tập hợp
    all_links.update(links)

    print(f"Đã thu thập {len(links)} link từ trang {page}")

# Ghi tất cả link vào file
with open("alphabooks_links.txt", "w", encoding="utf-8") as f:
    for link in all_links:
        f.write(link + "\n")

print(f"Tổng cộng thu thập được {len(all_links)} link sách.")


In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_alphabooks(url):
    response = requests.get(url)
    if response.status_code != 200:
        print("Failed to retrieve the webpage")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    tab_contents = soup.find_all('div', class_='tab-content')

    data = []
    for tab in tab_contents:
        tab_text = tab.get_text(separator=' ', strip=True)
        tables = tab.find_all('table')
        table_data = []

        for table in tables:
            headers = [header.get_text(strip=True) for header in table.find_all('th')]
            rows = []

            for row in table.find_all('tr')[1:]:  # Skip header row
                cells = row.find_all('td')
                row_data = {headers[i]: cells[i].get_text(strip=True) for i in range(len(cells))}
                rows.append(row_data)

            table_data.append(rows)

        data.append({
            "text": tab_text,
            "tables": table_data
        })

    return data

url = "https://www.alphabooks.vn/co-phieu-thuong-loi-nhuan-phi-thuong"
result = scrape_alphabooks(url)
print(result)


In [ ]:
import json
import requests
from bs4 import BeautifulSoup

def extract_itemprop_data(url):
    response = requests.get(url)
    if response.status_code != 200:
        print(f"❌ Không thể truy cập {url}, mã lỗi {response.status_code}")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    section = soup.find('section')

    if not section:
        print(f"⚠️ Không tìm thấy <section> trong {url}")
        return None

    data = {}
    for tag in section.find_all(attrs={"itemprop": True}):
        key = tag.get("itemprop")
        value = tag.get("content") or tag.text.strip()
        data[key] = value

    return data

def process_urls(file_input, file_output):
    with open(file_input, "r", encoding="utf-8") as f:
        urls = [line.strip() for line in f if line.strip()]

    results = []
    for url in urls:
        data = extract_itemprop_data(url)
        print(url)
        if data:
            results.append(data)
            print(data)

    with open(file_output, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    print(f"✅ Dữ liệu đã được lưu vào {file_output}")

# Chạy chương trình
process_urls("alphabooks_links.txt", "alphabooks_data.json")

https://www.alphabooks.vn/metaverse-vu-tru-ao-va-cuoc-cach-mang-hoa-van-vat
{'category': 'Tủ sách Công nghệ & Chuyển đổi số', 'url': 'https://www.alphabooks.vn/metaverse-vu-tru-ao-va-cuoc-cach-mang-hoa-van-vat', 'name': 'Metaverse: Vũ Trụ Ảo Và Cuộc Cách Mạng Hóa Vạn Vật', 'image': 'http://bizweb.dktcdn.net/thumb/grande/100/197/269/products/metaverse.png?v=1676814400697', 'description': "\n\n\n\t\n\t\n\n\n\n\t\n\tCông ty phát hành\n\t\n\tAlpha Books\n\n\n\t\n\tLoại bìa\n\t\n\tBìa mềm, tay gấp\n\n\n\t\n\tKhổ sách\n\t\n\t16 x 24 cm\n\n\n\t\n\tNhà xuất bản\n\t\n\tNhà xuất bản Thế giới\n\n\n\nMETAVERSE: VŨ TRỤ ẢO VÀ CUỘC CÁCH MẠNG HÓA VẠN VẬT\n\nA. NỘI DUNG CUỐN SÁCH:\xa0\n\nThuật ngữ “Metaverse” đột nhiên xuất hiện ở khắp mọi nơi, từ trang nhất của các tờ báo quốc gia và các xu hướng thời trang mới nhất cho đến kế hoạch của các công ty quyền lực nhất trong lịch sử. Nó đã và đang định hình các nền tảng chính sách của chính phủ Hoa Kỳ, Liên minh châu Âu và Đảng cộng sản Trung Quốc.\n\nNhưng

In [ ]:
import re
import json
from urllib.parse import urlparse

def clean_text(text):
    if not isinstance(text, str):
        return text
    return re.sub(r"\s+", " ", text).strip()

def extract_info_from_description(description):
    patterns = {
        "published_date": r"Ngày xuất bản\s*(\d{1,2}-\d{4})",
        "dimensions": r"Kích thước\s*([\d\.x ]+cm)",
        "pages": r"Số trang\s*(\d+)",
        "cover_type": r"Loại bìa\s*([^\n]+)"
    }
    extracted = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, description)
        if match:
            extracted[key] = match.group(1)
    return extracted

def convert_price(price_str):
    try:
        return int(re.sub(r"[^0-9]", "", price_str))
    except ValueError:
        return None

def validate_url(url):
    return url if urlparse(url).scheme in ["http", "https"] else None

def preprocess_book(data):
    data["name"] = clean_text(data.get("name", ""))
    data["category"] = clean_text(data.get("category", ""))
    data["url"] = validate_url(data.get("url", ""))
    data["image"] = validate_url(data.get("image", ""))

    extracted_info = extract_info_from_description(data.get("description", ""))
    data.update(extracted_info)
    data.pop("description", None)

    data["author"] = clean_text(data.get("brand", "")) or clean_text(data.get("author", ""))
    data["publisher"] = clean_text(data.get("publisher", "")) or "Alpha Books"

    data["price"] = convert_price(data.get("price", "0"))
    data["original_price"] = convert_price(data.get("priceSpecification", "0"))

    if data["original_price"] and data["original_price"] > data["price"]:
        data["discount"] = round((data["original_price"] - data["price"]) / data["original_price"] * 100, 2)
    else:
        data["discount"] = 0

    data["availability"] = "Còn hàng" if data.get("availability", "") == "Còn hàng" else "Hết hàng"

    data["rating"] = float(data.get("ratingValue", 0))
    data["rating_count"] = int(data.get("ratingCount", 0))
    data["best_rating"] = int(data.get("bestRating", 5))
    data["worst_rating"] = int(data.get("worstRating", 1))

    data["sku"] = clean_text(data.get("sku", ""))

    keys_to_keep = ["name", "category", "url", "image", "author", "publisher", "published_date", "dimensions", "pages",
                    "cover_type", "price", "original_price", "discount", "currency", "availability", "rating",
                    "rating_count", "best_rating", "worst_rating", "sku"]
    return {key: data[key] for key in keys_to_keep if key in data}

# Example usage
data = {'category': 'Tất cả', 'url': 'https://www.alphabooks.vn/warren-buffett-qua-trinh-hinh-thanh-mot-nha-tu-ban-my-tai-ban', 'name': 'Warren Buffett - Quá Trình Hình Thành Một Nhà Tư Bản Mỹ (Tái Bản)', 'image': 'http://bizweb.dktcdn.net/thumb/grande/100/197/269/products/qua-trinh-hinh-thanh-mot-nha-thu-ban-my.jpg?v=1515211740393', 'description': 'Mô tảThông Tin Chi Tiết\n\n\n\nCông ty phát hành\nAlphabooks\n\n\nNgày xuất bản\n2021-06-15 00:00:00\n\n\nKích thước\n16 x 24 cm\n\n\nDịch Giả\nMinh Diệu - Phương Lan\n\n\nLoại bìa\nBìa mềm\n\n\nSố trang\n616\n\n\nNhà xuất bản\nNhà Xuất Bản Công Thương\n\n\nWarren Buffett - Quá Trình Hình Thành Một Nhà Tư Bản Mỹ\xa0là câu chuyện thú vị về cuộc đời và triết lý đầu tư của nhà lựa chọn cổ phiếu thành công nhất nước Mỹ. Tác giả cuốn sách, phóng viên tờ Wall Street Journal, Roger Lowenstein đã chỉ ra rằng phương pháp đầu tư của Buffett là sự phản chiếu của nhũng giá trị cuộc sống mà ông luôn theo đuổi. Bằng cách vén lên tấm màn bí mật bao quanh con người này, Roger Lowenstein khám phá ra những phẩm chất đáng quý ở ông - nhẫn nại, trung thành, liêm chính, kiên định. Cuốn sách này lần theo mọi dấu vết của cuộc đời một nhà tư bản Mỹ, từ lúc đi giao báo cho đến khi trở thành nhà đầu tư vĩ đại với khôi tài sản khổng lồ luôn nằm trong top 3 của thế giới. Không chỉ là một tuyển tập những giai thoại về tài chính doanh nghiệp, cuốn sách này còn là câu chuyện đầy tính nhân bản khắc họa nên chân dung của một con người thành công nhất thế kỷ XX.\n"Có rất nhiều cuốn sách viết về Warren Buffett và chiến lược đẩu tư của ông, nhưng... đây là cuốn đáng đọc nhất." - Bill Gates, Harvard Business Review\n"Cuốn Buffett của Roger Lowenstein còn hơn cả một cuốn tiểu sử; đó thực sự là một lát cắt sống động của nền văn minh Hoa Kỳ." - Adam Smith\n\n"Một cuốn\xa0sách hay\xa0cho ai muốn tìm hiểu về Warren Buffett, nhà hiền triết của Omaha, bậc thầy về đầu tư tài chính, một người "kỳ dị" trong đầu tư, "bình dị" trong phong cách và "giản dị" trong cuộc sống, một tấm gương sống động của thành công, một người "cao" mà không "xa", có Tầm tài năng và nhân cách mà có cuộc sống rất Đời Thường. Chỉ cần đọc vài trang, bạn sẽ khó buông cuốn sách này xuống khi chưa đọc hết. Tôi thích cuốn sách này!" - Lý Trường Chiến, Chuyên gia kinh tế cao cấp - tư vấn tái cấu trúc, quản trị chiến lược & phát triển nguồn lực\n\xa0\nKể từ khi được xuất bản vào tháng Tám năm 1995, cuốn Buffett đã xuất hiện trong danh sách bán chạy nhất của các tờ Wall Street Journal, New York Times, San Francisco Chronicle, Los Angeles Times, Seattle Times, Newsday và Business Week. Buffet là một bức chân dung mang tính bước ngoặt của một nhân vật có một không hai của nước Mỹ - Warren Buffett.\nKhởi đầu từ con số không, chỉ đơn giản bằng cách chọn các cổ phiếu và các công ty để đầu tư, Warren Buffett tích lũy được một một tài sản ròng đồ sộ trị giá 10 tỷ USD, và hiện vẫn đang ngày càng tăng. Hồ sơ đầu tư đáng kinh ngạc của ông đã khiến ông trở thành một hình mẫu được sùng bái trong giới đầu tư, nổi tiếng với vẻ mâu thuẫn của mình: một tỷ phú với lối sống rất chừng mực, một nhà đầu tư thành công phi thường mà không phải ra vào thường xuyên Phố Wall.\nNhà báo Roger Lowenstein dựa trên ba năm tiếp xúc với gia đình, bạn bè, và các đồng nghiệp của Buffett để đưa ra bản ghi chép đầu tiên, chi tiết và chân thực về đời sống và sự nghiệp của con người này. Roger Loweinstein giải thích chiến lược đầu tư của Buffett - một triết lý dài hạn căn cứ vào việc mua cổ phần của các công ty được định giá thấp trên thị trường và giữ cho đến khi đạt được giá trị thực của chúng - và chỉ ra triết lý đó chính là sự phản chiếu những giá trị cuộc sống mà ông theo đuổi từ thời trai trẻ.\nMời các bạn đón đọc!', 'brand': 'Warren Buffett', 'sku': '8935251417272', 'offers': 'Còn hàng', 'supersededBy': 'Còn hàng', 'availability': 'Còn hàng', 'priceCurrency': 'VND', 'price': '239200', 'priceSpecification': '299000', 'priceValidUntil': '2099-01-01', 'review': 'Warren Buffett - Quá Trình Hình Thành Một Nhà Tư Bản Mỹ (Tái Bản)\n\n\nAlpha Books\n\n\n10 out of\n\t\t\t10', 'itemReviewed': 'Warren Buffett - Quá Trình Hình Thành Một Nhà Tư Bản Mỹ (Tái Bản)', 'author': 'Alpha Books', 'reviewRating': '10 out of\n\t\t\t10', 'ratingValue': '5', 'bestRating': '5', 'publisher': '', 'aggregateRating': '5', 'worstRating': '1', 'ratingCount': '1'}

processed_data = preprocess_book(data)
print(json.dumps(processed_data, indent=4, ensure_ascii=False))

{
    "name": "Warren Buffett - Quá Trình Hình Thành Một Nhà Tư Bản Mỹ (Tái Bản)",
    "category": "Tất cả",
    "url": "https://www.alphabooks.vn/warren-buffett-qua-trinh-hinh-thanh-mot-nha-tu-ban-my-tai-ban",
    "image": "http://bizweb.dktcdn.net/thumb/grande/100/197/269/products/qua-trinh-hinh-thanh-mot-nha-thu-ban-my.jpg?v=1515211740393",
    "author": "Warren Buffett",
    "publisher": "Alpha Books",
    "dimensions": "16 x 24 cm",
    "pages": "616",
    "cover_type": "Bìa mềm",
    "price": 239200,
    "original_price": 299000,
    "discount": 20.0,
    "availability": "Còn hàng",
    "rating": 5.0,
    "rating_count": 1,
    "best_rating": 5,
    "worst_rating": 1,
    "sku": "8935251417272"
}


In [ ]:
import re
import json
from urllib.parse import urlparse

def clean_text(text):
    if not isinstance(text, str):
        return text
    return re.sub(r"\s+", " ", text).strip()

def extract_info_from_description(description):
    patterns = {
        "published_date": r"Ngày xuất bản\s*(\d{1,2}-\d{4})",
        "dimensions": r"Kích thước\s*([\d\.x ]+cm)",
        "pages": r"Số trang\s*(\d+)",
        "cover_type": r"Loại bìa\s*([^\n]+)"
    }
    extracted = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, description)
        if match:
            extracted[key] = match.group(1)
            description = description.replace(match.group(0), "")  # Remove extracted info from description
    return extracted, clean_text(description)

def convert_price(price_str):
    try:
        return int(re.sub(r"[^0-9]", "", price_str))
    except ValueError:
        return None

def validate_url(url):
    return url if urlparse(url).scheme in ["http", "https"] else None

def preprocess_book(data):
    data["name"] = clean_text(data.get("name", ""))
    data["category"] = re.sub(r"\s+", "-", data.get("category", "").strip().lower())
    data["url"] = validate_url(data.get("url", ""))
    data["image"] = validate_url(data.get("image", ""))

    extracted_info, cleaned_description = extract_info_from_description(data.get("description", ""))
    data.update(extracted_info)
    data["description"] = cleaned_description  # Keep remaining description content

    data["author"] = clean_text(data.get("brand", "")) or clean_text(data.get("author", ""))
    data["publisher"] = clean_text(data.get("publisher", "")) or "Alpha Books"

    data["price"] = convert_price(data.get("price", "0"))
    data["original_price"] = convert_price(data.get("priceSpecification", "0"))

    if data["original_price"] and data["original_price"] > data["price"]:
        data["discount"] = round((data["original_price"] - data["price"]) / data["original_price"] * 100, 2)
    else:
        data["discount"] = 0

    data["availability"] = "Còn hàng" if data.get("availability", "") == "Còn hàng" else "Hết hàng"

    rating_value = data.get("ratingValue", 0)
    if isinstance(rating_value, str):
        rating_value = rating_value.replace(",", ".")  # Replace comma with period
    data["rating"] = float(rating_value)
    data["rating_count"] = int(data.get("ratingCount", 0))
    data["best_rating"] = int(data.get("bestRating", 5))
    data["worst_rating"] = int(data.get("worstRating", 1))

    data["sku"] = clean_text(data.get("sku", ""))

    # Convert pages to integer
    if "pages" in data:
        try:
            data["pages"] = int(data["pages"])
        except ValueError:
            data["pages"] = None

    # Split dimensions into length and width
    if "dimensions" in data:
        dims = re.findall(r"\d+", data["dimensions"])
        if len(dims) == 2:
            data["length"], data["width"] = map(int, dims)
        data.pop("dimensions", None)

    keys_to_keep = ["name", "category", "url", "image", "author", "publisher", "published_date", "length", "width", "pages",
                    "cover_type", "price", "original_price", "discount", "currency", "availability", "rating",
                    "rating_count", "best_rating", "worst_rating", "sku", "description"]
    return {key: data[key] for key in keys_to_keep if key in data}

# Load data from file
with open("alphabooks_data.json", "r", encoding="utf-8") as file:
    books_data = json.load(file)

# Process each book entry
cleaned_books = [preprocess_book(book) for book in books_data]

# Save cleaned data to new file
with open("clean_alphabooks_data.json", "w", encoding="utf-8") as file:
    json.dump(cleaned_books, file, indent=4, ensure_ascii=False)


In [ ]:
def process_text(text):
    # Loại bỏ "Mô Tả" khỏi text
    text = text.replace("Mô Tả", "")

    text = text.replace("Mô tả", "")

    # Tìm vị trí xuất hiện cuối cùng của "\n\n\n"
    last_index = text.rfind("\n\n\n")

    # Nếu tìm thấy, cắt bỏ phần trước đó
    if last_index != -1:
        text = text[last_index + 3:]

    return text

# Ví dụ sử dụng
text = "Mô Tả Đây là phần mô tả\n\n\nNội dung chính cần giữ lại."
result = process_text(text)
print(result)

Nội dung chính cần giữ lại.


In [ ]:
import json

with open("alphabooks_data.json", "r", encoding="utf-8") as file:
    books_data = json.load(file)

for book in books_data:
    print('##########',book['name'],'##########')
    print(process_text(book['description']))

########## Metaverse: Vũ Trụ Ảo Và Cuộc Cách Mạng Hóa Vạn Vật ##########
“Một cái nhìn kích thích tư duy về sự giao thoa đang phát triển của công nghệ, xã hội, bản chất con người và sự sáng tạo. Hướng dẫn khai sáng của Matthew Ball là cuốn sách cần phải đọc cho mọi người sáng tạo và công ty đang bắt đầu hành trình đến biên giới mới đó là Metaverse. Một thế giới mới được định hình bởi các cá nhân và được thúc đẩy bởi những trải nghiệm xã hội đáng kinh ngạc vừa bắt đầu. ” ― Kenichiro Yoshida, Giám đốc điều hành của Sony
“Matthew Ball đã viết những bài luận có sức ảnh hưởng lớn trên Metaverse trong nhiều năm nay, và ở đây anh ấy chắt lọc mọi thứ mình biết thành một hướng dẫn có thẩm quyền cho bất kỳ ai trong chúng ta. Cuốn sách này không chỉ ghi lại khoảnh khắc của chúng ta ― nó còn định hình tương lai của tập thể chúng ta. ” ―Taylor Lorenz, phóng viên mảng công nghệ tại Washington Post
'Metaverse' là một từ thông dụng mới, nhưng ít người có thể định nghĩa nó. Luận thuyết toàn diện của Ma

In [ ]:
import re
import json
from urllib.parse import urlparse

def clean_text(text):
    if not isinstance(text, str):
        return text
    return re.sub(r"\s+", " ", text).strip()

def extract_info_from_description(description):
    patterns = {
        "published_date": r"Ngày xuất bản\s*(\d{1,2}-\d{4})",
        "dimensions": r"Kích thước\s*([\d\.x ]+cm)",
        "pages": r"Số trang\s*(\d+)",
        "cover_type": r"Loại bìa\s*([^\n]+)"
    }
    extracted = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, description)
        if match:
            extracted[key] = match.group(1)
    return extracted

def convert_price(price_str):
    try:
        return int(re.sub(r"[^0-9]", "", price_str))
    except ValueError:
        return None

def validate_url(url):
    return url if urlparse(url).scheme in ["http", "https"] else None

def preprocess_book(data):
    data["name"] = clean_text(data.get("name", ""))
    data["category"] = clean_text(data.get("category", ""))
    data["url"] = validate_url(data.get("url", ""))
    data["image"] = validate_url(data.get("image", ""))

    extracted_info = extract_info_from_description(data.get("description", ""))
    data.update(extracted_info)
    data.pop("description", None)

    data["author"] = clean_text(data.get("brand", "")) or clean_text(data.get("author", ""))
    data["publisher"] = clean_text(data.get("publisher", "")) or "Alpha Books"

    data["price"] = convert_price(data.get("price", "0"))
    data["original_price"] = convert_price(data.get("priceSpecification", "0"))

    if data["original_price"] and data["original_price"] > data["price"]:
        data["discount"] = round((data["original_price"] - data["price"]) / data["original_price"] * 100, 2)
    else:
        data["discount"] = 0

    data["availability"] = "Còn hàng" if data.get("availability", "") == "Còn hàng" else "Hết hàng"

    rating_value = data.get("ratingValue", 0)
    if isinstance(rating_value, str):
        rating_value = rating_value.replace(",", ".")  # Replace comma with period
    data["rating"] = float(rating_value)
    data["rating_count"] = int(data.get("ratingCount", 0))
    data["best_rating"] = int(data.get("bestRating", 5))
    data["worst_rating"] = int(data.get("worstRating", 1))

    data["sku"] = clean_text(data.get("sku", ""))

    # Convert pages to integer
    if "pages" in data:
        try:
            data["pages"] = int(data["pages"])
        except ValueError:
            data["pages"] = None

    # Split dimensions into length and width
    if "dimensions" in data:
        dims = re.findall(r"\d+", data["dimensions"])
        if len(dims) == 2:
            data["length"], data["width"] = map(int, dims)
        data.pop("dimensions", None)

    keys_to_keep = ["name", "category", "url", "image", "author", "publisher", "published_date", "length", "width", "pages",
                    "cover_type", "price", "original_price", "discount", "currency", "availability", "rating",
                    "rating_count", "best_rating", "worst_rating", "sku"]
    return {key: data[key] for key in keys_to_keep if key in data}

# Example usage
data = {
    "category": "Tủ sách Tài chính - Đầu tư",
    "url": "https://www.alphabooks.vn/vanderbilt-tai-phiet-dau-tien-cua-nuoc-my",
    "name": "Vanderbilt - Tài Phiệt Đầu Tiên Của Nước Mỹ",
    "image": "http://bizweb.dktcdn.net/thumb/grande/100/197/269/products/vanderbilt-tai-phiet-dau-tien-cua-nuoc-my-a-04-copy.png?v=1697604346767",
    "description": "\n\n\n\tCông ty phát hành\n\tAlpha Books\n\n\n\tNgày xuất bản\n\t10 -2023\n\n\n\tKích thước\n\t16 x 24 cm\n\n\n\tLoại bìa\n\tBìa cứng, áo ôm\n\n\n\tSố trang\n\t884\n\n\n\tTác giả\n\t\n\tT.J. Stiles\n\n\n",
    "brand": "T.J. Stiles",
    "sku": "8935251420654",
    "price": "399200",
    "priceSpecification": "499000",
    "priceCurrency": "VND",
    "availability": "Còn hàng",
    "ratingValue": "2",
    "bestRating": "5",
    "worstRating": "1",
    "ratingCount": "1"
}

processed_data = preprocess_book(data)
print(json.dumps(processed_data, indent=4, ensure_ascii=False))


{
    "name": "Vanderbilt - Tài Phiệt Đầu Tiên Của Nước Mỹ",
    "category": "Tủ sách Tài chính - Đầu tư",
    "url": "https://www.alphabooks.vn/vanderbilt-tai-phiet-dau-tien-cua-nuoc-my",
    "image": "http://bizweb.dktcdn.net/thumb/grande/100/197/269/products/vanderbilt-tai-phiet-dau-tien-cua-nuoc-my-a-04-copy.png?v=1697604346767",
    "author": "T.J. Stiles",
    "publisher": "Alpha Books",
    "length": 16,
    "width": 24,
    "pages": 884,
    "cover_type": "Bìa cứng, áo ôm",
    "price": 399200,
    "original_price": 499000,
    "discount": 20.0,
    "availability": "Còn hàng",
    "rating": 2.0,
    "rating_count": 1,
    "best_rating": 5,
    "worst_rating": 1,
    "sku": "8935251420654"
}


In [ ]:

# Load data from file
with open("alphabooks_data.json", "r", encoding="utf-8") as file:
    books_data = json.load(file)

# Process each book entry
cleaned_books = [preprocess_book(book) for book in books_data]

# Save cleaned data to new file
with open("clean_alphabooks_data.json", "w", encoding="utf-8") as file:
    json.dump(cleaned_books, file, indent=4, ensure_ascii=False)


In [ ]:
import json
import requests
from bs4 import BeautifulSoup

def extract_itemprop_data(url):
    response = requests.get(url)
    if response.status_code != 200:
        print(f"❌ Không thể truy cập {url}, mã lỗi {response.status_code}")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    section = soup.find('section')

    if not section:
        print(f"⚠️ Không tìm thấy <section> trong {url}")
        return None

    data = {}
    for tag in section.find_all(attrs={"itemprop": True}):
        key = tag.get("itemprop")
        value = tag.get("content") or tag.text.strip()
        data[key] = value

    return data

def process_urls(file_input, file_output):
    with open(file_input, "r", encoding="utf-8") as f:
        urls = [line.strip() for line in f if line.strip()]

    results = []
    for url in urls:
        data = extract_itemprop_data(url)
        print(url)
        if data:
            results.append(data)
            print(data)

    with open(file_output, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    print(f"✅ Dữ liệu đã được lưu vào {file_output}")

# Chạy chương trình
process_urls("alphabooks_links.txt", "alphabooks_data.json")

https://www.alphabooks.vn/metaverse-vu-tru-ao-va-cuoc-cach-mang-hoa-van-vat
{'category': 'Tủ sách Công nghệ & Chuyển đổi số', 'url': 'https://www.alphabooks.vn/metaverse-vu-tru-ao-va-cuoc-cach-mang-hoa-van-vat', 'name': 'Metaverse: Vũ Trụ Ảo Và Cuộc Cách Mạng Hóa Vạn Vật', 'image': 'http://bizweb.dktcdn.net/thumb/grande/100/197/269/products/metaverse.png?v=1676814400697', 'description': "\n\n\n\t\n\t\n\n\n\n\t\n\tCông ty phát hành\n\t\n\tAlpha Books\n\n\n\t\n\tLoại bìa\n\t\n\tBìa mềm, tay gấp\n\n\n\t\n\tKhổ sách\n\t\n\t16 x 24 cm\n\n\n\t\n\tNhà xuất bản\n\t\n\tNhà xuất bản Thế giới\n\n\n\nMETAVERSE: VŨ TRỤ ẢO VÀ CUỘC CÁCH MẠNG HÓA VẠN VẬT\n\nA. NỘI DUNG CUỐN SÁCH:\xa0\n\nThuật ngữ “Metaverse” đột nhiên xuất hiện ở khắp mọi nơi, từ trang nhất của các tờ báo quốc gia và các xu hướng thời trang mới nhất cho đến kế hoạch của các công ty quyền lực nhất trong lịch sử. Nó đã và đang định hình các nền tảng chính sách của chính phủ Hoa Kỳ, Liên minh châu Âu và Đảng cộng sản Trung Quốc.\n\nNhưng